# Laboratório — Intervalos de confiança e incerteza da estimativa

[![Abrir no Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/joaopaulomirandamatias/ai-lab/blob/main/02-statistics/notebooks/14-intervalos-confianca-laboratorio.ipynb)

Este laboratório verifica intervalos analíticos, cobertura por simulação e bootstrap. Todas as fontes aleatórias usam seed fixa.

**Dependências:** Python 3.10+, NumPy, pandas, SciPy e Matplotlib.

## 1. Preparação

Usaremos uma função auxiliar para Wilson e quatro experimentos reproduzíveis:

1. intervalo t para a média;
2. Wald versus Wilson para proporção;
3. cobertura em repetições;
4. bootstrap para mediana e comparação pareada de modelos.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
from IPython.display import display
from scipy.stats import bootstrap, norm, t

SEED = 20260907
np.set_printoptions(precision=6, suppress=True)

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("SciPy:", scipy.__version__)
print("Seed:", SEED)

## 2. Intervalo t para a média

Para variância populacional desconhecida e dados independentes aproximadamente normais:

\[
\bar x \pm t_{1-\alpha/2,n-1}\frac{s}{\sqrt n}.
\]

Vamos calcular o exemplo de latência manualmente e conferir com \`scipy.stats.t.interval\`.

In [ ]:
latencias = np.array(
    [92, 88, 95, 101, 97, 90, 93, 105, 99, 94, 96, 91],
    dtype=float,
)
nivel = 0.95
alpha = 1 - nivel
n = latencias.size
media = latencias.mean()
desvio = latencias.std(ddof=1)
erro_padrao = desvio / np.sqrt(n)
t_critico = t.ppf(1 - alpha / 2, df=n - 1)
margem = t_critico * erro_padrao
ic_t_manual = (media - margem, media + margem)
ic_t_scipy = t.interval(
    confidence=nivel,
    df=n - 1,
    loc=media,
    scale=erro_padrao,
)

print(f"n={n}, média={media:.6f}, s={desvio:.6f}, SE={erro_padrao:.6f}")
print(f"t crítico={t_critico:.9f}")
print(f"IC 95% manual: [{ic_t_manual[0]:.6f}, {ic_t_manual[1]:.6f}] ms")
print(f"IC 95% SciPy:  [{ic_t_scipy[0]:.6f}, {ic_t_scipy[1]:.6f}] ms")

assert np.allclose(ic_t_manual, ic_t_scipy)
assert np.allclose(ic_t_manual, [91.9889005066, 98.1777661601])

### t versus normal padrão

Em amostras pequenas, o valor crítico t é maior porque a variância populacional foi estimada. A diferença diminui com os graus de liberdade.

In [ ]:
graus = np.array([2, 3, 5, 10, 20, 50, 100, 500])
criticos_t = t.ppf(0.975, graus)
z_critico = norm.ppf(0.975)

tabela_criticos = pd.DataFrame({
    "graus_de_liberdade": graus,
    "t_0,975": criticos_t,
    "excesso_sobre_z": criticos_t - z_critico,
})
display(tabela_criticos.round(6))

plt.figure(figsize=(8, 4))
plt.plot(graus, criticos_t, "o-", label="t crítico")
plt.axhline(z_critico, color="#dc2626", ls="--", label=f"z = {z_critico:.3f}")
plt.xscale("log")
plt.xlabel("graus de liberdade")
plt.ylabel("valor crítico bilateral de 95%")
plt.title("A distribuição t se aproxima da normal")
plt.legend()
plt.tight_layout()
plt.show()

## 3. Proporção: comparar Wald e Wilson

O intervalo de Wald pode sair do suporte \([0,1]\). Implementaremos Wilson diretamente para tornar explícita cada parcela da fórmula.

In [ ]:
def intervalo_wilson(sucessos, total, nivel=0.95):
    if total <= 0 or not 0 <= sucessos <= total:
        raise ValueError("Exija total > 0 e 0 <= sucessos <= total.")
    p_hat = sucessos / total
    z = norm.ppf(1 - (1 - nivel) / 2)
    denominador = 1 + z**2 / total
    centro = (p_hat + z**2 / (2 * total)) / denominador
    meia_largura = (
        z / denominador
        * np.sqrt(
            p_hat * (1 - p_hat) / total
            + z**2 / (4 * total**2)
        )
    )
    return centro - meia_largura, centro + meia_largura

sucessos, total = 8, 10
p_hat = sucessos / total
z = norm.ppf(0.975)
se_wald = np.sqrt(p_hat * (1 - p_hat) / total)
ic_wald = (p_hat - z * se_wald, p_hat + z * se_wald)
ic_wilson = intervalo_wilson(sucessos, total)

print(f"Estimativa: {p_hat:.3f}")
print(f"Wald 95%:   [{ic_wald[0]:.6f}, {ic_wald[1]:.6f}]")
print(f"Wilson 95%: [{ic_wilson[0]:.6f}, {ic_wilson[1]:.6f}]")

assert ic_wald[1] > 1
assert np.allclose(ic_wilson, [0.4901624715, 0.9433178485])
assert 0 <= ic_wilson[0] <= ic_wilson[1] <= 1

### O problema piora perto das fronteiras

Para zero sucessos, o erro-padrão plug-in de Wald vale zero e o intervalo degenera em \([0,0]\). Wilson ainda reconhece incerteza.

In [ ]:
linhas_proporcao = []
for sucessos_i in [0, 1, 5, 9, 10]:
    p_i = sucessos_i / 10
    se_i = np.sqrt(p_i * (1 - p_i) / 10)
    wald_i = (p_i - z * se_i, p_i + z * se_i)
    wilson_i = intervalo_wilson(sucessos_i, 10)
    linhas_proporcao.append({
        "sucessos": sucessos_i,
        "p_hat": p_i,
        "Wald_inf": wald_i[0],
        "Wald_sup": wald_i[1],
        "Wilson_inf": wilson_i[0],
        "Wilson_sup": wilson_i[1],
    })

df_proporcao = pd.DataFrame(linhas_proporcao)
display(df_proporcao.round(6))

## 4. Cobertura: a propriedade aparece em repetições

Simularemos 50 mil amostras normais de tamanho 12. Compararemos:

- **z conhecido:** usa o \(\sigma\) populacional, apenas como referência;
- **t correto:** estima \(\sigma\) por \(s\) e usa t;
- **z ingênuo:** estima \(\sigma\) por \(s\), mas mantém 1,96.

Esperamos cobertura próxima de 95% nos dois primeiros e subcobertura no terceiro.

In [ ]:
rng = np.random.default_rng(SEED)
repeticoes = 50_000
n_cobertura = 12
mu = 10.0
sigma = 4.0

amostras = rng.normal(mu, sigma, size=(repeticoes, n_cobertura))
medias = amostras.mean(axis=1)
desvios = amostras.std(axis=1, ddof=1)

meia_z_conhecido = z * sigma / np.sqrt(n_cobertura)
inf_z = medias - meia_z_conhecido
sup_z = medias + meia_z_conhecido

tcrit = t.ppf(0.975, df=n_cobertura - 1)
meia_t = tcrit * desvios / np.sqrt(n_cobertura)
inf_t = medias - meia_t
sup_t = medias + meia_t

meia_z_ingenuo = z * desvios / np.sqrt(n_cobertura)
inf_zi = medias - meia_z_ingenuo
sup_zi = medias + meia_z_ingenuo

coberturas = pd.DataFrame({
    "método": ["z com σ conhecido", "t com s estimado", "z com s estimado"],
    "cobertura": [
        np.mean((inf_z <= mu) & (mu <= sup_z)),
        np.mean((inf_t <= mu) & (mu <= sup_t)),
        np.mean((inf_zi <= mu) & (mu <= sup_zi)),
    ],
    "largura_média": [
        np.mean(sup_z - inf_z),
        np.mean(sup_t - inf_t),
        np.mean(sup_zi - inf_zi),
    ],
})
display(coberturas.round(6))

assert abs(coberturas.loc[0, "cobertura"] - 0.95) < 0.005
assert abs(coberturas.loc[1, "cobertura"] - 0.95) < 0.005
assert coberturas.loc[2, "cobertura"] < 0.94

In [ ]:
quantos = 100
indices = np.arange(quantos)
cobriu = (inf_t[:quantos] <= mu) & (mu <= sup_t[:quantos])

plt.figure(figsize=(10, 6))
for i in indices:
    cor = "#2563eb" if cobriu[i] else "#dc2626"
    plt.plot([inf_t[i], sup_t[i]], [i, i], color=cor, lw=1.4)
    plt.plot(medias[i], i, ".", color=cor)
plt.axvline(mu, color="black", ls="--", label="média verdadeira")
plt.xlabel("intervalo calculado")
plt.ylabel("réplica")
plt.title("Azul cobre o parâmetro; vermelho não cobre")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Entre os 100 intervalos exibidos, {cobriu.sum()} cobriram μ.")

## 5. Margem de erro e tamanho amostral

Para uma proporção planejada com \(p_0=0,5\), nível de 95% e margem de três pontos percentuais:

\[
n \geq \frac{z^2p_0(1-p_0)}{E^2}.
\]

O resultado deve ser arredondado para cima.

In [ ]:
p_planejado = 0.5
margem_desejada = 0.03
n_continuo = z**2 * p_planejado * (1 - p_planejado) / margem_desejada**2
n_planejado = int(np.ceil(n_continuo))

print(f"n antes do arredondamento: {n_continuo:.6f}")
print(f"n mínimo inteiro: {n_planejado}")
assert n_planejado == 1068

In [ ]:
n_grid = np.array([25, 50, 100, 200, 400, 800, 1600])
margens = z * np.sqrt(0.25 / n_grid)
referencia = margens[0] * np.sqrt(n_grid[0] / n_grid)

df_largura = pd.DataFrame({
    "n": n_grid,
    "margem_aproximada": margens,
    "referência_1/sqrt(n)": referencia,
})
display(df_largura.round(6))

plt.figure(figsize=(8, 4))
plt.loglog(n_grid, margens, "o-", label="margem para p=0,5")
plt.loglog(n_grid, referencia, "--", label="referência 1/√n")
plt.xlabel("n")
plt.ylabel("margem de erro")
plt.title("Precisão cresce com a raiz do tamanho amostral")
plt.legend()
plt.tight_layout()
plt.show()
assert np.allclose(margens, referencia)

## 6. Bootstrap da mediana em dados assimétricos

Geraremos 60 observações lognormais. A mediana é menos sensível à cauda que a média, mas seu erro-padrão não surge da fórmula da média. Calcularemos intervalos percentil e BCa com 9.999 réplicas.

A seed controla a reamostragem; ela não transforma uma amostra ruim em representativa.

In [ ]:
rng_dados = np.random.default_rng(SEED + 1)
tempos = rng_dados.lognormal(mean=0.0, sigma=0.8, size=60)
mediana_observada = np.median(tempos)

resultado_percentil = bootstrap(
    (tempos,),
    np.median,
    confidence_level=0.95,
    n_resamples=9_999,
    method="percentile",
    batch=1_000,
    rng=np.random.default_rng(SEED + 2),
)
resultado_bca = bootstrap(
    (tempos,),
    np.median,
    confidence_level=0.95,
    n_resamples=9_999,
    method="BCa",
    batch=1_000,
    rng=np.random.default_rng(SEED + 3),
)
ic_percentil = (
    resultado_percentil.confidence_interval.low,
    resultado_percentil.confidence_interval.high,
)
ic_bca = (
    resultado_bca.confidence_interval.low,
    resultado_bca.confidence_interval.high,
)

print(f"Mediana observada: {mediana_observada:.6f}")
print(f"IC percentil: [{ic_percentil[0]:.6f}, {ic_percentil[1]:.6f}]")
print(f"IC BCa:       [{ic_bca[0]:.6f}, {ic_bca[1]:.6f}]")
print(f"SE bootstrap: {resultado_bca.standard_error:.6f}")

assert ic_percentil[0] < mediana_observada < ic_percentil[1]
assert ic_bca[0] < mediana_observada < ic_bca[1]
assert resultado_bca.standard_error > 0

In [ ]:
replicas = resultado_percentil.bootstrap_distribution

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(tempos, bins=15, color="#93c5fd", edgecolor="white")
axes[0].axvline(mediana_observada, color="#dc2626", ls="--", label="mediana")
axes[0].set(xlabel="tempo", ylabel="frequência", title="Amostra assimétrica")
axes[0].legend()

axes[1].hist(replicas, bins=35, color="#99f6e4", edgecolor="white")
axes[1].axvline(ic_percentil[0], color="#7c3aed", ls=":")
axes[1].axvline(ic_percentil[1], color="#7c3aed", ls=":", label="IC percentil")
axes[1].axvline(mediana_observada, color="#dc2626", ls="--", label="mediana")
axes[1].set(xlabel="mediana bootstrap", ylabel="frequência", title="Distribuição bootstrap")
axes[1].legend()
plt.tight_layout()
plt.show()

## 7. Comparação pareada de dois classificadores

Os modelos A e B são avaliados nos mesmos 2.000 casos. Cada caso pertence a um estado: ambos erram, apenas A acerta, apenas B acerta ou ambos acertam. Reamostrar a diferença por caso preserva o pareamento.

In [ ]:
rng_modelos = np.random.default_rng(SEED + 4)
n_teste = 2_000
# Estados: ambos erram, apenas A acerta, apenas B acerta, ambos acertam.
estados = rng_modelos.choice(4, size=n_teste, p=[0.12, 0.05, 0.08, 0.75])
acerto_a = np.isin(estados, [1, 3]).astype(float)
acerto_b = np.isin(estados, [2, 3]).astype(float)
diferenca_por_caso = acerto_b - acerto_a

acc_a = acerto_a.mean()
acc_b = acerto_b.mean()
delta = diferenca_por_caso.mean()

resultado_pareado = bootstrap(
    (diferenca_por_caso,),
    np.mean,
    confidence_level=0.95,
    n_resamples=9_999,
    method="BCa",
    batch=1_000,
    rng=np.random.default_rng(SEED + 5),
)
ic_delta = (
    resultado_pareado.confidence_interval.low,
    resultado_pareado.confidence_interval.high,
)

print(f"Acurácia A: {acc_a:.6f}")
print(f"Acurácia B: {acc_b:.6f}")
print(f"Δ(B-A):     {delta:.6f}")
print(f"IC 95% pareado para Δ: [{ic_delta[0]:.6f}, {ic_delta[1]:.6f}]")

assert np.isclose(delta, acc_b - acc_a)
assert ic_delta[0] < delta < ic_delta[1]

### O estimando precisa estar explícito

O intervalo anterior varia os casos de teste e mantém os modelos congelados. Ele não inclui novos splits, inicializações, ajustes de hiperparâmetros nem drift temporal.

Para medir o pipeline completo, essas etapas devem ser repetidas dentro de um desenho externo apropriado, sem usar o teste final para selecionar o modelo.

## 8. Verificações finais

In [ ]:
checagens = {
    "intervalo t manual = SciPy": np.allclose(ic_t_manual, ic_t_scipy),
    "Wilson respeita [0,1]": 0 <= ic_wilson[0] <= ic_wilson[1] <= 1,
    "cobertura z próxima de 95%": abs(coberturas.loc[0, "cobertura"] - 0.95) < 0.005,
    "cobertura t próxima de 95%": abs(coberturas.loc[1, "cobertura"] - 0.95) < 0.005,
    "z ingênuo subcobre": coberturas.loc[2, "cobertura"] < 0.94,
    "planejamento arredondado para cima": n_planejado == 1068,
    "bootstrap da mediana não degenerado": resultado_bca.standard_error > 0,
    "diferença pareada preservada": np.isclose(delta, acc_b - acc_a),
}

for nome, passou in checagens.items():
    print(f"{'PASSOU' if passou else 'FALHOU'} — {nome}")
assert all(checagens.values())

## Desafios

1. Troque o nível de confiança de 95% para 90% e 99%. Compare larguras.
2. Simule a cobertura de Wald e Wilson para \(p=0,05\), \(n=20\).
3. Quadruplicate o tamanho do teste dos classificadores e observe a largura do intervalo da diferença.
4. Gere dados com 50 usuários e dez observações correlacionadas por usuário. Compare bootstrap de linhas e de usuários.
5. Repita o treino de um classificador simples dentro de cada réplica externa e explique por que o estimando mudou.

Na Aula 15, a distribuição amostral será usada para formular e avaliar hipóteses com erros I/II e poder.